# Fase 2 — Preprocesamiento y selección de variables

El pronóstico se emite al cierre de un día **D**, a las **23:59**, para el consumo de frío del día calendario siguiente. Partimos de `dataset_trabajo.csv`, generado con `python -m src.correccion_fechas`. Las fases de preprocesamiento de esta notebook y del script usan las mismas funciones.

## 2.1 Limpieza y corte cronológico

Los Excel se unen por fecha y hora. Ante fechas repetidas se prioriza el libro más reciente y se conserva la primera jornada de las planillas con bloques apilados. Se descartan días incompletos, cierres imposibles y fallas de reinicio o de medidor. El MAD y el percentil de saltos se estiman con el primer año, hasta el **30/06/2021**, antes de Validation y Test. El P99,9 de incrementos de calibración resultó **4.956 kW**. Lo redondeamos a la centena superior (**5.000 kW**) para que diferencias de pocos kW no definan un descarte. La limpieza excluye 18 días y reproduce byte por byte el `dataset_trabajo.csv` que había obtenido el equipo.

Para definir las particiones buscamos primero el valor de **D+1** por fecha exacta. Si falta ese cierre o el de D es inválido, quitamos el par del conjunto supervisado. Un cierre válido sin etiqueta sigue sirviendo como historial para las ventanas posteriores.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error

# Permite abrir la notebook desde notebooks/ o desde la raíz del repo.
raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.auxiliar_functions import DATA_DIR, SEMILLA, huella, verificar_checksums
from src.feature_engineering import (
    CLIMA_CSV,
    PreparadorFeatures,
    clima_historico,
    columnas_numericas,
    features_temporales,
    ordenar_diario,
)
from src.features_planta import COLUMNAS_BASE, FeaturesPlanta
from src.preprocessing_pipeline import (
    DEFAULT_TARGET_COL,
    aggregate_daily_data,
    load_cleaned_data,
    prepare_target,
    run_pipeline,
    temporal_split,
)
from src.seleccion_variables import seleccionar_variables

# Chequeamos las entradas antes de analizar o modificar los datos procesados.
verificar_checksums()
with open(DATA_DIR / "checksums.json", encoding="utf-8") as archivo:
    registro = json.load(archivo)
ruta_trabajo = DATA_DIR / "processed" / "dataset_trabajo.csv"
assert huella(ruta_trabajo) == registro["data/processed/dataset_trabajo.csv"]
assert huella(CLIMA_CSV) == registro["data/reference/clima_tecate_openmeteo.csv"]
print("Entradas verificadas. Semilla:", SEMILLA)

Verificación de integridad completada: 3 archivo(s) coinciden.
Entradas verificadas. Semilla: 42


In [2]:
# Ejemplo mínimo con un cierre nulo y un hueco en el calendario.
ejemplo = pd.DataFrame({
    "DIA": pd.to_datetime(["2020-01-01", "2020-01-02", "2020-01-03", "2020-01-05", "2020-01-06"]),
    "Frio": [110, np.nan, 130, 150, 160],
})
pares_ejemplo, informe_ejemplo = prepare_target(ejemplo, target_col="Frio")
assert pares_ejemplo["DIA"].dt.strftime("%Y-%m-%d").tolist() == ["2020-01-05"]
assert pares_ejemplo["y"].tolist() == [160]
display(pares_ejemplo)

,DIA,Frio,y
0,2020-01-05,150.0,160.0


In [3]:
# Usamos el mismo corte para todo lo que requiera un ajuste.
horario, informe_horario = load_cleaned_data()
diario, informe_diario = aggregate_daily_data(horario)
pares, informe_target = prepare_target(diario)
columnas_corte = ["DIA", DEFAULT_TARGET_COL, "y"]
division = temporal_split(pares[columnas_corte])[-1]
train = division["dataframes"]["train_df"].set_index("DIA")
validation = division["dataframes"]["val_df"].set_index("DIA")

print("Filas horarias y columnas:", horario.shape)
print("Días con cierre:", len(diario))
print("Pares sin etiqueta válida:", informe_target["discarded_observations"])
print("Pares supervisados:", len(pares))
resumen_cortes = {}
for nombre in ["train", "validation", "test"]:
    info = division[nombre]
    resumen_cortes[nombre] = [info["date_start"], info["date_end"], info["observations"]]
display(pd.DataFrame.from_dict(resumen_cortes, orient="index", columns=["Inicio D", "Fin D", "Filas"]))

Filas horarias y columnas: (29296, 421)
Días con cierre: 1172
Pares sin etiqueta válida: 32
Pares supervisados: 1140


,Inicio D,Fin D,Filas
train,2020-07-01,2022-10-26,798
validation,2022-10-27,2023-04-29,171
test,2023-04-30,2023-10-24,171


El último D de Train es el **26/10/2022**. Su etiqueta se conoce al cierre del **27/10**, primer D de Validation. En ese momento ya podemos ajustar el modelo y emitir el pronóstico del 28/10. Conservamos este criterio de disponibilidad al pasar de Validation al Test.

Para las variables operativas partimos de hojas de producción, energía, agua, aire, gas/vapor y CO₂. Mantenemos los criterios del equipo (**20 %** máximo de faltantes, **|ρ| ≥ 0,40** con `y` y descarte por colinealidad **|ρ| ≥ 0,90**), medidos en Train. Revisamos la lista más amplia por dominio para evitar que el ranking exploratorio de toda la serie determine quién puede competir. Excluimos medidores que contienen o casi duplican el cierre de Frío, que ya está disponible como `frio_hoy_d`.

Un valor negativo o mayor a **5 × P99** de Train se considera inválido para estos totalizadores. Rellenamos hasta **tres días calendario** hacia adelante y luego usamos la mediana de Train.

In [4]:
# Se ajustan las reglas con las 798 filas supervisadas de Train.
historial_train = diario.loc[diario["DIA"] <= train.index.max()]
historial_val = diario.loc[diario["DIA"] <= validation.index.max()]
preparador = PreparadorFeatures().fit(historial_train, train["y"])

# Incorporamos el historial anterior para que Validation no empiece con ventanas vacías.
candidatas = preparador.transform(historial_val)
X_train = candidatas.loc[train.index]
X_val = candidatas.loc[validation.index]
tabla_operativas = preparador.tabla_operativas_
print("Variables operativas evaluadas:", len(tabla_operativas))
print("Representantes operativas retenidas:", len(preparador.operativas_))
display(tabla_operativas.loc[preparador.operativas_].round(3))

# Contamos reparaciones sobre Train y verificamos que sus candidatas queden finitas.
base_train = ordenar_diario(historial_train).loc[train.index]
operativas_crudas = columnas_numericas(base_train, preparador.operativas_)
extremos = preparador.limpieza_.named_steps["extremos"]
operativas_filtradas = extremos.transform(operativas_crudas)
print("Lecturas operativas imposibles en Train:", int(extremos.reemplazos_.sum()))
print("Celdas operativas a imputar en Train:", int(operativas_filtradas.isna().sum().sum()))
assert np.isfinite(X_train).all().all()
assert np.isfinite(X_val).all().all()

Variables operativas evaluadas: 169
Representantes operativas retenidas: 28
Lecturas operativas imposibles en Train: 9
Celdas operativas a imputar en Train: 9


,faltantes_train,spearman_y_train,spearman_frio_hoy_train,decision
KW Trafo 11_Totalizadores Energia,0.000,0.710,0.817,Elegida
Agua Planta (Hl)_Consolidado Agua,0.000,0.688,0.779,Elegida
KW Cond 5. 6 y 9_Totalizadores Energia,0.000,0.674,0.710,Elegida
ET Servicios (Mj)_Consolidado GasVapor,0.000,-0.594,-0.591,Elegida
KW Trafo 9_Totalizadores Energia,0.000,0.590,0.648,Elegida
Vapor Servicio (Kg)_Consolidado GasVapor,0.000,-0.577,-0.577,Elegida
KW Trafo 10_Totalizadores Energia,0.000,0.556,0.716,Elegida
Aire Producido (M3)_Consolidado Aire,0.000,0.543,0.637,Elegida
Agua Envasado (Hl)_Consolidado Agua,0.000,0.530,0.562,Elegida
Elaboracion (Kw)_Consolidado EE,0.001,0.526,0.668,Elegida


## 2.2 Variables derivadas

Los rezagos representan **D−1, D−2 y D−7 calendario**. Las ventanas de 3, 7 y 14 días incluyen D y exigen, respectivamente, **2, 4 y 7** cierres observados. El desvío semanal exige cuatro. Imputamos faltantes de predictoras con estadísticas de Train y calculamos la diferencia de frío desde el mismo rezago usado por las demás features.

Medimos mes, día de semana y fin de semana sobre Train. En el modelo se mantiene mes e índice estacional por la magnitud de su efecto. Las variables semanales se muestran como diagnóstico.

In [5]:
# Comparamos cuánto de la variación de y se asocia con cada variable calendario.
calendario = pd.DataFrame(index=train.index)
calendario["mes"] = calendario.index.month
calendario["dia_semana"] = calendario.index.day_of_week
calendario["fin_de_semana"] = (calendario.index.day_of_week >= 5).astype(int)
calendario["y"] = train["y"]
media_train = calendario["y"].mean()
variacion_total = ((calendario["y"] - media_train) ** 2).sum()

for columna in ["mes", "dia_semana", "fin_de_semana"]:
    resumen = calendario.groupby(columna)["y"].agg(["size", "mean"])
    variacion_grupos = (resumen["size"] * (resumen["mean"] - media_train) ** 2).sum()
    print(f"{columna}: eta² = {100 * variacion_grupos / variacion_total:.2f} %")

base_temporal = features_temporales(ordenar_diario(historial_val))
rezagos_train = base_temporal.loc[train.index]
tabla_temporal = pd.DataFrame({
    "nulos_train_antes_de_imputar": rezagos_train.isna().sum(),
    "spearman_y_train": rezagos_train.corrwith(train["y"], method="spearman"),
})
display(tabla_temporal.round(3))

mes: eta² = 68.01 %
dia_semana: eta² = 0.46 %
fin_de_semana: eta² = 0.07 %


,nulos_train_antes_de_imputar,spearman_y_train
frio_hoy_d,0,0.831
frio_lag_1,22,0.736
frio_lag_2,27,0.719
frio_lag_7,34,0.715
frio_roll_mean_3,6,0.817
frio_roll_mean_7,3,0.825
frio_roll_mean_14,6,0.827
frio_roll_std_7,3,0.105
mes_numero,0,-0.085
calor_estacional,0,0.788


### Planta, metas e interacciones

Conservamos el análisis de Genaro sobre ratios, eficiencia energética e interacción producción × clima. La eficiencia compara **EE Frio / Hl** con la meta de la misma hoja y unidad. Si falta la meta usamos la mediana de las metas positivas para ese mes, calculada en Train. El indicador `meta_frio_imputada` sirve para auditar el cálculo.

En las candidatas del modelo, tratamos `Frio / Servicios` fuera de `[0, 1]` como una inconsistencia de medidor y lo imputamos. La variable de agua de condensadores tiene un tramo largo sin datos. El clima histórico queda como diagnóstico de 2.2 porque el archivo original carece de la hora de publicación de cada observación. La interacción apta para el modelo usa producción de Bodega × índice estacional calculado con el mes de D.

In [6]:
# Reproducimos el diagnóstico climático con la copia fija de Tecate.
clima = clima_historico()
clima_train = clima.loc[train.index]
display(clima_train.corrwith(train["y"], method="spearman").sort_values(ascending=False).round(3))

# Aplicamos el cálculo de planta de Genaro solo al tramo histórico de Train.
entrada_diagnostico = ordenar_diario(historial_train).join(clima)
planta_diagnostico = FeaturesPlanta()
planta_diagnostico.fit(entrada_diagnostico.loc[train.index])
derivadas = planta_diagnostico.transform(entrada_diagnostico).loc[train.index]
tabla_planta = pd.DataFrame({
    "spearman_y_train": derivadas.corrwith(train["y"], method="spearman"),
    "faltantes_train_pct": 100 * derivadas.isna().mean(),
})
display(tabla_planta.round(3))
print("Metas sustituidas en Train:", int(derivadas["meta_frio_imputada"].sum()))

frio_d = base_train[COLUMNAS_BASE["frio"]]
servicios = base_train[COLUMNAS_BASE["servicios"]]
ratio_bruto = frio_d / servicios.where(servicios > 0)
print("Cocientes Frío/Servicios mayores a 1 en Train:", int((ratio_bruto > 1).sum()))

Metas sustituidas en Train: 348
Cocientes Frío/Servicios mayores a 1 en Train: 2


radiacion_roll7       0.772
radiacion_roll3       0.761
radiacion_mj_m2       0.726
temp_aparente_c       0.558
temp_min_c            0.532
temp_media_roll7      0.531
temp_media_roll3      0.525
temp_media_c          0.515
temp_media_lag1       0.512
temp_max_roll3        0.507
temp_max_c            0.487
amplitud_termica_c    0.035
dtype: float64

,spearman_y_train,faltantes_train_pct
ratio_frio_servicios,0.715,0.000
ratio_bodega_elaboracion,0.038,0.000
ratio_aguacond_aguaplanta,-0.121,43.609
meta_frio_imputada,0.033,0.000
eficiencia_frio_vs_meta,0.308,0.000
bodega_x_temp_media_c,0.689,0.000
bodega_x_radiacion_roll7,0.767,0.000


## 2.3 Selección de variables

Ajustamos **LASSO** con variables estandarizadas y **Random Forest** para obtener importancia por reducción de error cuadrático. Ambos ven solo Train. Fijamos `alpha = 100` para LASSO y un máximo de **12** variables según importancia del bosque antes de evaluar los subconjuntos.

Comparamos todas las candidatas, las de LASSO, las del bosque y la intersección. Usamos el mismo **Ridge** con `alpha = 10` y escala ajustada en Train para comparar el efecto de las columnas. Gana el MAE más bajo en **Validation**, con desempate por menos columnas a una precisión de 0,01 kW. Todas las alternativas incluyen Frío de hoy, que es el punto de partida para persistencia. El Test se reserva para la evaluación final de la Fase 3.

In [7]:
# Baselines sobre las mismas fechas de Validation que la comparación de subconjuntos.
y_val = validation["y"]
persistencia = base_temporal.loc[y_val.index, "frio_hoy_d"]
media_movil = base_temporal.loc[y_val.index, "frio_roll_mean_7"]
assert persistencia.notna().all() and media_movil.notna().all()
print(f"Persistencia en Validation: {mean_absolute_error(y_val, persistencia):.2f} kW")
print(f"Media móvil de 7 días en Validation: {mean_absolute_error(y_val, media_movil):.2f} kW")

columnas_finales, evidencia, comparacion = seleccionar_variables(X_train, train["y"], X_val, y_val)
display(comparacion)
display(evidencia.loc[columnas_finales].round(4))
print("Ganador en Validation:", comparacion.loc[0, "subconjunto"])
print("Cantidad de variables finales:", len(columnas_finales))

Persistencia en Validation: 2727.13 kW
Media móvil de 7 días en Validation: 2395.32 kW
Ganador en Validation: Random Forest
Cantidad de variables finales: 12


,subconjunto,columnas,mae_validation
0,Random Forest,12,2149.43
1,Intersección,9,2227.85
2,LASSO,22,2286.60
3,Todas,43,2286.66


,coeficiente_lasso,importancia_bosque,seleccion_lasso,seleccion_bosque,interseccion
ET Servicios (Mj)_Consolidado GasVapor,-184.1526,0.0053,True,True,True
KW Secador Kaeser_Totalizadores Energia,307.5762,0.0055,True,True,True
KW Linea 3_Totalizadores Energia,0.0000,0.0070,False,True,False
Red Paste L4_Consolidado Agua,187.0913,0.0058,True,True,True
KW Trafo 12_Totalizadores Energia,280.7422,0.0050,True,True,True
frio_hoy_d,1663.0015,0.2913,True,True,True
frio_roll_mean_3,0.0000,0.0319,False,True,False
frio_roll_mean_7,12.2911,0.4110,True,True,True
frio_roll_mean_14,2551.7536,0.0931,True,True,True
frio_diff_1,382.4963,0.0098,True,True,True


El subconjunto se elige por la tabla anterior. Los baselines sirven para ubicar la escala del error. La comparación de la Fase 2 usa Validation para tomar una decisión, por lo que la mejora debe volver a medirse en el Test final bajo el protocolo de la consigna.

## 2.4 Exportación, escala y linaje

Ejecutamos el mismo script que se puede llamar desde terminal. Guardamos el CSV final y cada partición en unidades originales. El escalador aprende de `X_train` dentro de un `Pipeline`. Para la auditoría comprobamos la lista ganadora, el corte, la igualdad de los datos exportados y sus huellas SHA-256.

In [8]:
# Ejecuta la misma ruta que python -m src.preprocessing_pipeline.
resultado = run_pipeline()
assert resultado["X_train"].columns.tolist() == columnas_finales
pd.testing.assert_frame_equal(resultado["comparacion_subconjuntos"], comparacion)

# Verificamos que el escalador aprendió de las columnas numéricas de Train.
scaler = resultado["scaler_pipeline"].named_steps["scaler"]
np.testing.assert_allclose(scaler.mean_, resultado["X_train"].mean().to_numpy())
assert scaler.n_samples_seen_ == len(resultado["X_train"])

# El CSV debe conservar los valores originales y y debe corresponder a D+1.
ruta_final = DATA_DIR / "processed" / "dataset_final.csv"
exportado = pd.read_csv(ruta_final, parse_dates=["DIA"])
pd.testing.assert_frame_equal(exportado, resultado["df_final"], check_dtype=False)
frio_por_dia = diario.set_index("DIA")[DEFAULT_TARGET_COL]
y_esperado = (exportado["DIA"] + pd.Timedelta(days=1)).map(frio_por_dia)
np.testing.assert_allclose(exportado["y"], y_esperado)
assert exportado["DIA"].is_unique and exportado["y"].notna().all()

# Leemos los bytes de los CSV recién guardados y los contrastamos con el registro.
with open(DATA_DIR / "checksums.json", encoding="utf-8") as archivo:
    registro_final = json.load(archivo)
for nombre, ruta in resultado["summary"]["exported_paths"].items():
    assert huella(Path(ruta)) == registro_final[f"data/processed/{nombre}"]
print("Alineación D+1, shapes, escala de Train y hashes verificados.")

SECCIÓN 2.4 — RESUMEN DE PREPARACIÓN Y VERSIONADO DE DATOS
Días iniciales                  : 1172
Días con registro 23:59         : 1172
Días descartados (sin 23:59)    : 0
Observaciones descartadas (D+1) : 32 (discontinuidad/último día=32, cierre inválido D+1=0, solo cierre D inválido=0)
Observaciones con target válido : 1140
--------------------------------------------------------------------------------
TRAIN      | n= 798 (70.00%) | X=[798, 12] | y=[798] | 2020-07-01 -> 2022-10-26
VALIDATION | n= 171 (15.00%) | X=[171, 12] | y=[171] | 2022-10-27 -> 2023-04-29
TEST       | n= 171 (15.00%) | X=[171, 12] | y=[171] | 2023-04-30 -> 2023-10-24
--------------------------------------------------------------------------------
Checksums SHA-256 generados:
  dataset_diario_base.csv  : dc8b215bb583e9c082512a2d7488b57616382e28ca7b7e9a170f364df1c00a3b
  dataset_final.csv        : 2d023012ed97d340f30d79d57744c71fd8575ac03b0b56347bad3817932b6228
  train.csv                : b0f285a1fce335b4762f51a